# pandas-基本功能

## 教學目標

這份教學的目標是介紹基本的 pandas 功能，並學習快速操作大量的資料。

## 適用對象

適用於有程式基礎，且擁有 python 和 `numpy` 基礎的學生。

若沒有先學過 Python，建議先補充 Python 入門語法。

若沒有先學過 `numpy`，請參考 [numpy-基本功能](./numpy-基本功能.ipynb) 教學。

## 原教材的執行時間與測試環境

以下是 `original` 中原教材的測試記錄，精簡版的執行時間會因內容與環境而不同。

本教學全部執行時間約為 1.3843801021575928 秒。

|測試環境|名稱|
|-|-|
|主機板|X570 AORUS ELITE|
|處理器|AMD Ryzen 7 3700X 8-Core Processor|
|記憶體|Kingston KHX3200C16D4/16GX|
|硬碟|Seagate ST1000DM003-1ER1|
|顯示卡|GeForce RTX 2080|
|作業系統|Ubuntu 18.04 LTS|

本精簡版適合已學過 vibe coding 的學生，請參考 [使用說明](./README.md)。

## 大綱

- 簡介與安裝
- 讀取與檢查資料
- 選取資料與處理缺失值
- 衍生欄位、摘要與準備 X／y
- 練習

## 簡介

根據 [pandas 官方網站](https://pandas.pydata.org/)（v1.0）：

> pandas is a fast, powerful, flexible and easy to use open source data analysis and manipulation tool,
built on top of the Python programming language.
>
> pandas 是基於 python 開發的一個快速、強大、具有彈性且容易使用的開源資料分析與操作工具。

`pandas` 提供**讀取**或**輸出**以下資料格式的功能：

- CSV
- JSON
- HTML
- Microsoft Excel
- OpenDocument Excel
- SQL databases
- pickle
- Google BigQuery

讀取資料後，`pandas` 能夠對資料進行以下的操作：

- **索引（Index**）與**選取（Select）** 資料
- **合併（Merge）**、**結合（Join）** 與**串接（Concatenate）** 資料
- **統計（Statistic）** 與**視覺化（Visualization）**

## 安裝

透過 `pip` 安裝 `pandas`：

```sh
pip install pandas
```

## 1. 讀取與檢查

DataFrame 是表格：一列（row）是一筆樣本，一欄（column）是一個欄位；單一欄是 Series。
使用隨附的 **12 筆模擬氣象資料**，不用下載外部檔案。
從本 notebook 所在的 `simplified` 目錄執行。

In [ ]:
import pandas as pd

df = pd.read_csv('./data/weather_sample.csv')
display(df.head())
df.info()
display(df.describe())

**檢查重點：**資料有幾筆？數字欄位是否讀成數字？有沒有缺失或不合理的範圍？
`describe()` 的 count 不含缺失值，不能直接當作總樣本數。

## 2. 選欄位與篩樣本

`df[欄位列表]` 選特徵；`df.loc[條件, 欄位]` 選符合條件的資料。
`iloc` 按位置取值，例如 `df.iloc[:3]` 是前三筆。

In [ ]:
features = df[['Temperature', 'Humidity']]
warm_days = df.loc[df['Temperature'] > 22, ['Day', 'Temperature']]
print("特徵表 shape：", features.shape)
display(warm_days)

## 3. 缺失值：先觀察，再決定策略

`isna().sum()` 檢查各欄缺失筆數；`fillna()` 可填補，`dropna()` 可刪除缺失樣本。
本例用中位數填補氣溫，只示範操作；填補或刪除都可能改變資料分布。
**實際 ML 要先切分資料，再由訓練集學習填補值**，驗證／測試集使用同一個值。

In [ ]:
print(df.isna().sum())
clean = df.copy()
clean['Temperature'] = clean['Temperature'].fillna(df['Temperature'].median())
print("填補後剩餘缺失數：", clean['Temperature'].isna().sum())

### 小練習 1：訓練集與驗證集的缺失值

```python
train = df.iloc[:6].copy()
valid = df.iloc[6:].copy()
median = train['Temperature'].median()
valid['Temperature'] = valid['Temperature'].fillna(median)
print(median)
print(valid.loc[valid['Day'] == 7, 'Temperature'])
```

這段程式會輸出什麼？請說明第 7 天的氣溫如何得到，以及若先合併訓練集和驗證集再計算中位數，對模型評估有何影響。請在下方空白儲存格以 `#` 註解作答。

<details>
<summary>參考答案</summary>

訓練集氣溫的中位數為 21.5，程式將驗證集中缺失的第 7 天氣溫填成 21.5，因此兩次輸出的數值都是 21.5。原始 `df` 不受影響，因為 `train` 和 `valid` 是複本。

若將驗證集也納入計算，填補規則便使用了驗證資料的資訊，造成資料洩漏；評估結果可能無法反映模型面對未知資料的表現。

</details>

## 4. 衍生欄位與摘要

簡單公式可直接套用整欄；需要修改資料時，使用明確的欄位指派或 `.loc`。
不必為每筆資料另寫函式。摘要用來理解資料與檢查處理結果。

In [ ]:
clean['Temperature_F'] = clean['Temperature'] * 9 / 5 + 32
display(clean[['Day', 'Temperature', 'Temperature_F']].head())
display(clean.groupby('Weather')['Temperature'].agg(['count', 'mean']))

## 5. 準備 X 與 y

`X` 是用來預測的特徵，`y` 是答案。兩者必須對應同一批樣本、相同順序。
假設任務是用氣溫與濕度預測是否下雨，答案 `Rain` 不可同時放進 `X`。
類別欄位的編碼與訓練／驗證切分在 scikit-learn 篇處理。

In [ ]:
X = clean[['Temperature', 'Humidity']]
y = clean['Rain']
print("X:", X.shape, "y:", y.shape)
print("轉成 NumPy 後：", X.to_numpy().shape)

### 根據缺失值選取樣本

原教材介紹 `notna()` 找出有資料的位置，以及用條件篩選資料。
多個條件可用 `&` 組合；篩選後再從同一張表選出特徵與答案，便能保留對應的索引。
`index` 可用來檢查樣本順序；分組的 `count` 會統計該欄非缺失的筆數。

In [ ]:
has_temperature = df['Temperature'].notna()
has_humidity = df['Humidity'].notna()
example_rows = df.loc[has_temperature & has_humidity]
print('篩選後索引：', example_rows.index)
print(example_rows.groupby('Rain')['Temperature'].agg(['count']))

### 小練習 2：篩選後的特徵與標籤

```python
valid = df['Temperature'].notna() & df['Humidity'].notna()
selected = df.loc[valid]
features = selected[['Temperature', 'Humidity']]
labels = selected['Rain']
print(features.shape, labels.shape)
print(features.index, labels.index)
```

請解釋輸出的 shape 和兩組索引各代表什麼，並說明若把最後一行中的 `labels` 改成 `df['Rain']`，特徵與標籤會產生什麼問題。請在下方空白儲存格以 `#` 註解作答。

<details>
<summary>參考答案</summary>

`features` 是 `(11, 2)`，表示保留下來的 11 筆樣本各有兩個特徵；`labels` 是 `(11,)`，有 11 個對應標籤。兩者取自同一個 `selected`，所以索引相同、順序一致；原始索引 6（第 7 天）因氣溫缺失而不在其中。

若使用 `df['Rain']`，標籤仍有 12 筆，而特徵只有 11 筆，且標籤包含被移除樣本，兩者不再對齊，不能直接配對訓練。

</details>

## 練習：下需求，也驗收結果

請 AI 用原始 `df` 找出高於平均氣溫的日期與總天數，說明缺失值如何處理。
再檢查：結果是否全都高於平均？總天數是否等於篩選後的列數？

In [ ]:
# 參考解答：mean 忽略缺失值；缺失氣溫不會符合大於條件。
mean_temp = df['Temperature'].mean()
above_mean = df.loc[df['Temperature'] > mean_temp, ['Day', 'Temperature']]
display(above_mean)
print("平均氣溫：", mean_temp, "高於平均的天數：", len(above_mean))